<a href="https://colab.research.google.com/github/wtryab-re/data-preprocessing/blob/main/Encoding_Categorical_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [220]:
!pip install -q numpy matplotlib seaborn pandas

In [221]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

Label Encoding is useful when there’s a natural ordinal relationship.

One-Hot Encoding is ideal for nominal categories but can lead to high dimensionality.

Target Encoding and Binary Encoding are better for high-cardinality features but come with the risk of overfitting.

Frequency Encoding is a quick way to retain the distribution of the categorical data.

In [222]:
train = pd.read_csv("train.csv")
train.shape

(891, 12)

In [223]:
test = pd.read_csv("test.csv")
test.shape

(418, 11)

In [224]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [225]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [226]:
train.nunique()

,0
PassengerId,891
Survived,2
Pclass,3
Name,891
Sex,2
Age,88
SibSp,7
Parch,7
Ticket,681
Fare,248


In [227]:
test.nunique()

,0
PassengerId,418
Pclass,3
Name,418
Sex,2
Age,79
SibSp,7
Parch,8
Ticket,363
Fare,169
Cabin,76


In [228]:
df = pd.concat([train, test])
df.shape

(1309, 12)

In [229]:
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
413,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S
417,1309,NaN,3,"Peter, Master. Michael J",male,NaN,1,1,2668,22.3583,NaN,C


In [230]:
df.isna().sum()

,0
PassengerId,0
Survived,418
Pclass,0
Name,0
Sex,0
Age,263
SibSp,0
Parch,0
Ticket,0
Fare,1


In [231]:
df.dropna(subset=["Survived"], axis=0, inplace=True)
df.shape

(891, 12)

In [232]:
df.drop(["Name", "Ticket", "Cabin", "PassengerId"], inplace = True, axis=1)

In [233]:
X = df.drop(["Survived"], axis=1)
y = df["Survived"]

In [234]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [235]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((712, 7), (179, 7), (712,), (179,))

In [236]:
num_cols = ["SibSp", "Parch", "Fare", "Age"]
cat_cols = list(filter(lambda x: x not in num_cols and x not in ["PassengerId", "Survived"], df.columns))
print(cat_cols,num_cols)

['Pclass', 'Sex', 'Embarked'] ['SibSp', 'Parch', 'Fare', 'Age']


In [237]:
for col in cat_cols:
  print(col, df[col].unique())

Pclass [3 1 2]
Sex ['male' 'female']
Embarked ['S' 'C' 'Q' nan]


pclass is ordinal so label encoding

sex is nominal but binary so label encoding

embarked is nominal so one hot encoding

binary = label_encoding

multi_class/ordered = label_encoding

multiclass/nominal = One hot encoding

In [238]:
fillers = {"X_train":X_train["Embarked"].mode()[0],
           "X_test": X_test["Embarked"].mode()[0]}

In [239]:
X_train.fillna({"Embarked": X_train["Embarked"].mode()[0]}, inplace=True)

In [240]:
cols_for_le = ["Sex", "Pclass"]
cols_for_one = ["Embarked"]

In [241]:
label_encoder = {}
for col in cols_for_le:
  le = LabelEncoder()
  label_encoder[col] = le.fit(X_train[col])
  X_train[col] = le.transform(X_train[col])
  X_test[col] = le.transform(X_test[col])
label_encoder

{'Sex': LabelEncoder(), 'Pclass': LabelEncoder()}

One hot Encoding

To represent 3 categories, you need only TWO columns
00, 01, 10

drop just ignores the first category creation because need n-1 columns

handle_unknown = ignores the unknown category and makes it 00, incase ur testing data has unknown category

sparse_output = returns a numpy array instead of a sparse matrix

this type of encoding can easily increase dimensionality

In [242]:
one_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

In [243]:
one_encoder.fit(X_train[cols_for_one])

OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

In [244]:
Xo = one_encoder.transform(X_train[cols_for_one])
Xteo = one_encoder.transform(X_test[cols_for_one])

In [245]:
one_hot_cols = one_encoder.get_feature_names_out(cols_for_one)
one_hot_cols

array(['Embarked_Q', 'Embarked_S'], dtype=object)

In [246]:
X_train = pd.concat([X_train, pd.DataFrame(Xo, columns=one_hot_cols)], axis=1)
X_test = pd.concat([X_test, pd.DataFrame(Xteo, columns=one_hot_cols)], axis=1)

In [247]:
X_train.drop(cols_for_one, axis=1, inplace=True)
X_test.drop(cols_for_one, axis=1, inplace=True)

In [251]:
train_df = pd.concat([X_train, y_train], axis=1)
train_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Survived
692,2.0,1.0,NaN,0.0,0.0,56.4958,0.0,1.0,1.0
481,1.0,1.0,NaN,0.0,0.0,0.0000,0.0,1.0,0.0
527,0.0,1.0,NaN,0.0,0.0,221.7792,0.0,1.0,0.0
855,2.0,0.0,18.0,0.0,1.0,9.3500,NaN,NaN,1.0
801,1.0,0.0,31.0,1.0,1.0,26.2500,NaN,NaN,1.0


In [252]:
test_df = pd.concat([X_test, y_test], axis=1)
test_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Survived
565,2.0,1.0,24.0,2.0,0.0,24.1500,NaN,NaN,0.0
160,2.0,1.0,44.0,0.0,1.0,16.1000,0.0,0.0,0.0
553,2.0,1.0,22.0,0.0,0.0,7.2250,NaN,NaN,1.0
860,2.0,1.0,41.0,2.0,0.0,14.1083,NaN,NaN,0.0
241,2.0,0.0,NaN,1.0,0.0,15.5000,NaN,NaN,1.0


In [253]:
train_df.to_csv("train_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)